In [1]:
"""
Multi-Agent AI Workflow
========================
Orchestrator delegates to 3 specialist agents:
  - Researcher   → understands and breaks down the task
  - Calculator   → handles all math with tool use  
  - Writer       → produces the final response

Then an Evaluator agent scores the output.
"""

import json
import re
import anthropic

with open("config.json", "r") as f:
    config = json.load(f)

API_KEY = config["API_KEY"]
MODEL = config["MODEL"]
client  = anthropic.Anthropic(api_key=API_KEY)


"""
Multi-Agent AI Workflow — Dynamic Orchestration
=================================================
Orchestrator asks Claude which agents are needed for the task,
then only calls those agents.

Agents available:
  - researcher   → breaks down the task
  - calculator   → handles math
  - fact_checker → verifies factual claims
  - writer       → produces the final response
  - evaluator    → scores the output
"""


# ── Shared helpers ────────────────────────────────────────────────────────────

def call_claude(system, user_message, tools=None):
    kwargs = {
        "model":      MODEL,
        "max_tokens": 1000,
        "system":     system,
        "messages":   [{"role": "user", "content": user_message}],
    }
    if tools:
        kwargs["tools"] = tools
    return client.messages.create(**kwargs)

def get_text(response):
    return "\n".join(b.text for b in response.content if b.type == "text")

def get_tool_use(response):
    return next((b for b in response.content if b.type == "tool_use"), None)

def print_agent(name, content):
    print(f"\n{'─'*60}")
    print(f"  🤖 {name}")
    print(f"{'─'*60}")
    print(content)


# ── Tool ─────────────────────────────────────────────────────────────────────

CALCULATOR_TOOL = {
    "name": "calculator",
    "description": "Evaluates a math expression and returns the numeric result.",
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {"type": "string", "description": 'e.g. "142 * 37"'}
        },
        "required": ["expression"]
    }
}

def run_calculator(expression):
    safe = "".join(c for c in expression if c in "0123456789+-*/.() %")
    try:
        return str(round(eval(safe), 4))
    except Exception as e:
        return f"Error: {e}"


# ── Specialist Agents ─────────────────────────────────────────────────────────

def researcher_agent(task: str) -> str:
    response = call_claude(
        system="""You are a research agent. Your only job is to:
1. Understand what the user is asking
2. Identify all sub-tasks (math, writing, facts, etc.)
3. Return a clear structured brief for other agents
Be concise and specific.""",
        user_message=task
    )
    result = get_text(response)
    print_agent("Researcher Agent", result)
    return result

def calculator_agent(brief: str) -> str:
    response = call_claude(
        system="You are a calculator agent. If there is any math in the brief, call the calculator tool.",
        user_message=brief,
        tools=[CALCULATOR_TOOL]
    )
    tool_use = get_tool_use(response)
    if tool_use:
        expr   = tool_use.input["expression"]
        result = run_calculator(expr)
        output = f'calculator("{expr}") = {result}'
    else:
        output = "No math found."
    print_agent("Calculator Agent", output)
    return output

def fact_checker_agent(content: str) -> str:
    response = call_claude(
        system="""You are a fact-checking agent. Review the content for factual accuracy.
Flag anything that seems incorrect or unverifiable.
If everything looks good, say 'All facts check out.'""",
        user_message=content
    )
    result = get_text(response)
    print_agent("Fact Checker Agent", result)
    return result

def writer_agent(task: str, context: dict) -> str:
    context_str = "\n\n".join(f"{k}: {v}" for k, v in context.items() if v)
    response = call_claude(
        system="You are a writer agent. Using the task and all context provided, write a clear, complete, friendly response to the user.",
        user_message=f"Task: {task}\n\n{context_str}"
    )
    result = get_text(response)
    print_agent("Writer Agent", result)
    return result

def evaluator_agent(task: str, final_response: str) -> dict:
    response = call_claude(
        system="""You are an evaluator agent. Score the response 1-5 on correctness, completeness, and clarity.
Return JSON only: {"score": N, "reason": "..."}""",
        user_message=f"Task: {task}\n\nResponse: {final_response}"
    )
    raw   = get_text(response)
    match = re.search(r'\{.*?\}', raw, re.DOTALL)
    result = json.loads(match.group()) if match else {"score": None, "reason": raw}
    print_agent("Evaluator Agent", f"Score: {result['score']}/5 — {result['reason']}")
    return result


# ── Dynamic Router ────────────────────────────────────────────────────────────

AVAILABLE_AGENTS = ["researcher", "calculator", "fact_checker", "writer", "evaluator"]

def route(task: str) -> list[str]:
    """Ask Claude which agents are needed. Always includes writer + evaluator."""
    response = call_claude(
        system=f"""You are a routing agent. Given a task, decide which agents are needed from this list:
{AVAILABLE_AGENTS}

Rules:
- Always include "writer" and "evaluator"
- Include "researcher" for complex or multi-part tasks
- Include "calculator" only if arithmetic is required
- Include "fact_checker" if the task involves factual claims, history, or science

Return JSON only: {{"agents": ["agent1", "agent2", ...]}}
Order them in the sequence they should run.""",
        user_message=task
    )
    raw   = get_text(response)
    match = re.search(r'\{.*?\}', raw, re.DOTALL)
    if not match:
        return ["researcher", "writer", "evaluator"]   # safe default

    agents = json.loads(match.group()).get("agents", [])
    # Validate — only allow known agent names
    agents = [a for a in agents if a in AVAILABLE_AGENTS]

    print(f"\n  📋 Routing decision: {' → '.join(agents)}")
    return agents


# ── Orchestrator ──────────────────────────────────────────────────────────────

def orchestrator(task: str) -> dict:
    print(f"\n{'═'*60}")
    print(f"  ORCHESTRATOR received: {task}")
    print(f"{'═'*60}")

    # Step 1: Decide which agents to call
    agents = route(task)

    # Step 2: Run each agent, passing context forward
    context = {}

    for agent in agents:
        if agent == "researcher":
            context["research_brief"] = researcher_agent(task)

        elif agent == "calculator":
            brief = context.get("research_brief", task)
            context["math_result"] = calculator_agent(brief)

        elif agent == "fact_checker":
            # fact check whatever we have so far
            to_check = context.get("research_brief", task)
            context["fact_check"] = fact_checker_agent(to_check)

        elif agent == "writer":
            context["final_response"] = writer_agent(task, context)

        elif agent == "evaluator":
            final = context.get("final_response", "")
            context["score"] = evaluator_agent(task, final)

    print(f"\n{'═'*60}")
    score = context.get("score", {})
    print(f"  ORCHESTRATOR done. Final score: {score.get('score')}/5")
    print(f"{'═'*60}\n")

    return context


# ── Run it ────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    tasks = [
        "Calculate 142 × 37, then tell me a fun fact about that number.",   # → researcher + calculator + writer + evaluator
        "What are 3 steps to learn Python?",                                # → researcher + writer + evaluator
        "When did World War 2 end and what caused it?",                     # → researcher + fact_checker + writer + evaluator
    ]

    for task in tasks:
        orchestrator(task)
        print("\n" + "═"*60 + "\n")


════════════════════════════════════════════════════════════
  ORCHESTRATOR received: Calculate 142 × 37, then tell me a fun fact about that number.
════════════════════════════════════════════════════════════

  📋 Routing decision: calculator → researcher → fact_checker → writer → evaluator

────────────────────────────────────────────────────────────
  🤖 Calculator Agent
────────────────────────────────────────────────────────────
calculator("142 * 37") = 5254

────────────────────────────────────────────────────────────
  🤖 Researcher Agent
────────────────────────────────────────────────────────────
## Research Brief

### User Request
Calculate **142 × 37**, then provide a fun fact about the resulting number.

---

### Sub-Tasks

**Sub-Task 1: Math Calculation**
- Operation: Multiply 142 × 37
- Method: Standard multiplication
  - 142 × 30 = 4,260
  - 142 × 7 = 994
  - 4,260 + 994 = **5,254**
- ✅ Result: **5,254**

---

**Sub-Task 2: Fun Fact Research**
- Target number: **5,254**
-